In [ ]:
!pip install catboost -q

import os
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from catboost import CatBoostRegressor, CatBoostClassifier

# =====================================================
# 1. CSV PATH (Kaggle)
# =====================================================
POSSIBLE_PATHS = [
    "/kaggle/input/water-potability/water_potability.csv",
    "/kaggle/input/water_quality_and_potability/water_potability.csv",
]

def find_csv():
    for p in POSSIBLE_PATHS:
        if Path(p).exists():
            print("Using dataset file:", p)
            return p
    raise FileNotFoundError("Could not find water_potability.csv under /kaggle/input")

CSV_PATH = find_csv()


def score_ph(x):
    if pd.isna(x): return np.nan
    if 6.5 <= x <= 8.5: return 10.0
    if 5.5 <= x < 6.5: return (x - 5.5) / (6.5 - 5.5) * 10.0
    if 8.5 < x <= 10.0: return (10.0 - x) / (10.0 - 8.5) * 10.0
    return 0.0

def score_hardness(x):
    if pd.isna(x): return np.nan
    if x <= 200: return 10.0
    if 200 < x <= 600: return (600 - x) / (600 - 200) * 10.0
    return 0.0

def score_tds(x):
    if pd.isna(x): return np.nan
    if x <= 500: return 10.0
    if 500 < x <= 2000: return (2000 - x) / (2000 - 500) * 10.0
    return 0.0

def score_chloramines(x):
    if pd.isna(x): return np.nan
    if x <= 2: return 10.0
    if 2 < x <= 4: return (4 - x) / (4 - 2) * 10.0
    return 0.0

def score_sulfate(x):
    if pd.isna(x): return np.nan
    if x <= 200: return 10.0
    if 200 < x <= 400: return (400 - x) / (400 - 200) * 10.0
    return 0.0

def score_conductivity(x):
    if pd.isna(x): return np.nan
    if x <= 400: return 10.0
    if 400 < x <= 1000: return (1000 - x) / (1000 - 400) * 10.0
    return 0.0

def score_organic_carbon(x):
    if pd.isna(x): return np.nan
    if x <= 2: return 10.0
    if 2 < x <= 8: return (8 - x) / (8 - 2) * 10.0
    return 0.0

def score_trihalomethanes(x):
    if pd.isna(x): return np.nan
    if x <= 20: return 10.0
    if 20 < x <= 100: return (100 - x) / (100 - 20) * 10.0
    return 0.0

def score_turbidity(x):
    if pd.isna(x): return np.nan
    if x <= 1: return 10.0
    if 1 < x <= 5: return (5 - x) / (5 - 1) * 10.0
    return 0.0


WEIGHTS = {
    "ph_score": 1.4,
    "Hardness_score": 1.0,
    "Solids_score": 1.2,
    "Chloramines_score": 1.0,
    "Sulfate_score": 1.0,
    "Conductivity_score": 1.0,
    "Organic_carbon_score": 1.0,
    "Trihalomethanes_score": 1.2,
    "Turbidity_score": 1.2,
}


print("\nLoading dataset from:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))


COL_RENAMES = {}
for c in df.columns:
    low = c.lower().replace(" ", "_")
    if low == "organic_carbon" or "organic" in low:
        COL_RENAMES[c] = "Organic_carbon"
    if "trih" in low:
        COL_RENAMES[c] = "Trihalomethanes"
    if low == "ph":
        COL_RENAMES[c] = "ph"
    if "hard" in low:
        COL_RENAMES[c] = "Hardness"
    if "solid" in low or "tds" in low:
        COL_RENAMES[c] = "Solids"
    if "chlor" in low:
        COL_RENAMES[c] = "Chloramines"
    if "sulf" in low:
        COL_RENAMES[c] = "Sulfate"
    if "conduct" in low:
        COL_RENAMES[c] = "Conductivity"
    if "turb" in low:
        COL_RENAMES[c] = "Turbidity"
    if "potab" in low:
        COL_RENAMES[c] = "Potability"

if COL_RENAMES:
    df = df.rename(columns=COL_RENAMES)

required = [
    "ph","Hardness","Solids","Chloramines","Sulfate","Conductivity",
    "Organic_carbon","Trihalomethanes","Turbidity","Potability"
]
for c in required:
    if c not in df.columns:
        print("Adding missing column filled with NaN:", c)
        df[c] = np.nan


print("\nComputing rule-based per-parameter scores...")
df["ph_score"]              = df["ph"].apply(score_ph)
df["Hardness_score"]        = df["Hardness"].apply(score_hardness)
df["Solids_score"]          = df["Solids"].apply(score_tds)
df["Chloramines_score"]     = df["Chloramines"].apply(score_chloramines)
df["Sulfate_score"]         = df["Sulfate"].apply(score_sulfate)
df["Conductivity_score"]    = df["Conductivity"].apply(score_conductivity)
df["Organic_carbon_score"]  = df["Organic_carbon"].apply(score_organic_carbon)
df["Trihalomethanes_score"] = df["Trihalomethanes"].apply(score_trihalomethanes)
df["Turbidity_score"]       = df["Turbidity"].apply(score_turbidity)

def compute_aggregate(row):
    total = 0.0
    for k, w in WEIGHTS.items():
        v = row.get(k, np.nan)
        if pd.isna(v):
            v = 0.0
        total += v * w
    return total / sum(WEIGHTS.values())

df["aggregate_rule_score"] = df.apply(compute_aggregate, axis=1)
print("Aggregate score summary:")
print(df["aggregate_rule_score"].describe())



# basic interactions
df["ph_x_hardness"]      = df["ph"] * df["Hardness"]
df["organic_x_turbidity"] = df["Organic_carbon"] * df["Turbidity"]
df["solids_ratio"]       = df["Solids"] / (df["Hardness"] + 1)

df["log_solids"]         = np.log1p(df["Solids"])
df["log_conductivity"]   = np.log1p(df["Conductivity"])

# buckets
df["ph_bucket"] = pd.cut(
    df["ph"],
    bins=[0, 6, 7, 8.5, 14],
    labels=[0, 1, 2, 3]
)

df["organic_bucket"] = pd.cut(
    df["Organic_carbon"],
    bins=4,
    labels=False
)

df["ph_bucket"]      = pd.to_numeric(df["ph_bucket"], errors="coerce").fillna(0).astype(int)
df["organic_bucket"] = pd.to_numeric(df["organic_bucket"], errors="coerce").fillna(0).astype(int)

# additional interactions
df["hardness_x_conductivity"] = df["Hardness"] * df["Conductivity"]
df["chloramine_x_trihalo"]    = df["Chloramines"] * df["Trihalomethanes"]
df["organic_x_sulfate"]       = df["Organic_carbon"] * df["Sulfate"]
df["solid_density"]           = df["Solids"] / (df["Conductivity"] + 1)

# indices
df["danger_index"] = (
    df["Trihalomethanes"] * 0.25 +
    df["Chloramines"]      * 0.20 +
    df["Organic_carbon"]   * 0.15 +
    df["Turbidity"]        * 0.10
)

df["quality_index"] = (
    df["ph_bucket"] +
    df["organic_bucket"] +
    df["solids_ratio"]
)


FEATURES = [
    # original
    "ph","Hardness","Solids","Chloramines","Sulfate","Conductivity",
    "Organic_carbon","Trihalomethanes","Turbidity",

    # engineered
    "ph_x_hardness",
    "organic_x_turbidity",
    "solids_ratio",
    "log_solids",
    "log_conductivity",
    "ph_bucket",
    "organic_bucket",

    # new
    "hardness_x_conductivity",
    "chloramine_x_trihalo",
    "organic_x_sulfate",
    "solid_density",
    "danger_index",
    "quality_index",
]

X    = df[FEATURES].copy()
y_reg = df["aggregate_rule_score"].copy()
y_clf = df["Potability"].copy()


X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)
print("\nTrain/test sizes:", X_train.shape, X_test.shape)

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print("NaNs in X_train_prep:", np.isnan(X_train_prep).sum())
print("NaNs in y_reg_train:", np.isnan(y_reg_train).sum())

print("\nRunning 5-Fold Cross-Validation...")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

reg_mae_scores = []
clf_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\n--- Fold {fold} ---")

    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_reg_train_fold, y_reg_val_fold = y_reg.iloc[train_idx], y_reg.iloc[val_idx]
    y_clf_train_fold, y_clf_val_fold = y_clf.iloc[train_idx], y_clf.iloc[val_idx]

    X_train_prep_fold = preprocessor.fit_transform(X_train_fold)
    X_val_prep_fold   = preprocessor.transform(X_val_fold)

    reg_fold = CatBoostRegressor(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        verbose=False
    )
    clf_fold = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False
    )

    reg_fold.fit(X_train_prep_fold, y_reg_train_fold)
    clf_fold.fit(X_train_prep_fold, y_clf_train_fold)

    val_reg_pred  = reg_fold.predict(X_val_prep_fold)
    val_clf_proba = clf_fold.predict_proba(X_val_prep_fold)[:,1]

    fold_mae = mean_absolute_error(y_reg_val_fold, val_reg_pred)
    fold_auc = roc_auc_score(y_clf_val_fold, val_clf_proba)

    reg_mae_scores.append(fold_mae)
    clf_auc_scores.append(fold_auc)

    print(f"Fold MAE: {fold_mae:.4f}")
    print(f"Fold AUC: {fold_auc:.4f}")

print("\n==== Cross-Validation Summary ====")
print(f"Regressor MAE: {np.mean(reg_mae_scores):.4f}")
print(f"Classifier AUC: {np.mean(clf_auc_scores):.4f}")


reg = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False
)
clf = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=False
)

print("\nTraining final models on train split...")
reg.fit(X_train_prep, y_reg_train)
clf.fit(X_train_prep, y_clf_train)


y_reg_pred = reg.predict(X_test_prep)
mae  = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = mean_squared_error(y_reg_test, y_reg_pred, squared=False)
r2   = r2_score(y_reg_test, y_reg_pred)

print(f"\nRegressor evaluation: MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

y_clf_proba = clf.predict_proba(X_test_prep)[:,1]
y_clf_pred  = clf.predict(X_test_prep)

print("\nClassifier report:")
print(classification_report(y_clf_test, y_clf_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_clf_test, y_clf_proba))
print("Confusion matrix:")
print(confusion_matrix(y_clf_test, y_clf_pred))


OUT_DIR = Path("/kaggle/working/model_artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, OUT_DIR / "preprocessor.joblib")
joblib.dump(reg,          OUT_DIR / "regressor.joblib")
joblib.dump(clf,          OUT_DIR / "classifier.joblib")

df.to_csv(OUT_DIR / "data_with_scores.csv", index=False)

print("\nSaved artifacts to", OUT_DIR)
print("Files:")
for f in sorted(OUT_DIR.iterdir()):
    print(" -", f.name)


def build_feature_row(
    ph, Hardness, Solids, Chloramines, Sulfate,
    Conductivity, Organic_carbon, Trihalomethanes, Turbidity
):
    """Create a one-row DataFrame with ALL engineered features."""
    row = {
        "ph": ph,
        "Hardness": Hardness,
        "Solids": Solids,
        "Chloramines": Chloramines,
        "Sulfate": Sulfate,
        "Conductivity": Conductivity,
        "Organic_carbon": Organic_carbon,
        "Trihalomethanes": Trihalomethanes,
        "Turbidity": Turbidity,
    }
    temp = pd.DataFrame([row])

    temp["ph_x_hardness"]      = temp["ph"] * temp["Hardness"]
    temp["organic_x_turbidity"] = temp["Organic_carbon"] * temp["Turbidity"]
    temp["solids_ratio"]       = temp["Solids"] / (temp["Hardness"] + 1)

    temp["log_solids"]       = np.log1p(temp["Solids"])
    temp["log_conductivity"] = np.log1p(temp["Conductivity"])

    temp["ph_bucket"] = pd.cut(
        temp["ph"],
        bins=[0, 6, 7, 8.5, 14],
        labels=[0, 1, 2, 3]
    )
    temp["organic_bucket"] = pd.cut(
        temp["Organic_carbon"],
        bins=4,
        labels=False
    )
    temp["ph_bucket"]      = pd.to_numeric(temp["ph_bucket"], errors="coerce").fillna(0).astype(int)
    temp["organic_bucket"] = pd.to_numeric(temp["organic_bucket"], errors="coerce").fillna(0).astype(int)

    temp["hardness_x_conductivity"] = temp["Hardness"] * temp["Conductivity"]
    temp["chloramine_x_trihalo"]    = temp["Chloramines"] * temp["Trihalomethanes"]
    temp["organic_x_sulfate"]       = temp["Organic_carbon"] * temp["Sulfate"]
    temp["solid_density"]           = temp["Solids"] / (temp["Conductivity"] + 1)

    temp["danger_index"] = (
        temp["Trihalomethanes"] * 0.25 +
        temp["Chloramines"]      * 0.20 +
        temp["Organic_carbon"]   * 0.15 +
        temp["Turbidity"]        * 0.10
    )
    temp["quality_index"] = (
        temp["ph_bucket"] +
        temp["organic_bucket"] +
        temp["solids_ratio"]
    )

    return temp[FEATURES]


Using dataset file: /kaggle/input/water-potability/water_potability.csv

Loading dataset from: /kaggle/input/water-potability/water_potability.csv
Dataset shape: (3276, 10)
Columns: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']

Computing rule-based per-parameter scores...
Aggregate score summary:
count    3276.000000
mean        3.763633
std         0.758170
min         1.743036
25%         3.135476
50%         3.828540
75%         4.367471
max         5.963279
Name: aggregate_rule_score, dtype: float64

Train/test sizes: (2620, 22) (656, 22)
NaNs in X_train_prep: 0
NaNs in y_reg_train: 0

Running 5-Fold Cross-Validation...

--- Fold 1 ---
Fold MAE: 0.0352
Fold AUC: 0.7039

--- Fold 2 ---
Fold MAE: 0.0399
Fold AUC: 0.6921

--- Fold 3 ---
Fold MAE: 0.0388
Fold AUC: 0.6761

--- Fold 4 ---
Fold MAE: 0.0400
Fold AUC: 0.6538

--- Fold 5 ---
Fold MAE: 0.0369
Fold AUC: 0.6976

==== Cross-Validation Summa

In [ ]:
import joblib
import numpy as np

BASE = "/kaggle/input/water-quality-backend-model/scikitlearn/full-model/1"

# Load models
preprocessor = joblib.load(f"{BASE}/preprocessor.joblib")
regressor = joblib.load(f"{BASE}/regressor.joblib")
classifier = joblib.load(f"{BASE}/classifier.joblib")

print("Models loaded successfully!")


Models loaded successfully!


In [ ]:
import pandas as pd
import numpy as np

FEATURES = [
    # original features
    'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity',
    'Organic_carbon', 'Trihalomethanes', 'Turbidity',

    'ph_x_hardness',
    'organic_x_turbidity',
    'solids_ratio',
    'log_solids',
    'log_conductivity',
    'ph_bucket',
    'organic_bucket',
    'hardness_x_conductivity',
    'chloramine_x_trihalo',
    'organic_x_sulfate',
    'solid_density',
    'danger_index',
    'quality_index'
]

def build_feature_row(ph, Hardness, Solids, Chloramines, Sulfate,
                      Conductivity, Organic_carbon, Trihalomethanes, Turbidity):
    # Base dataframe with original 9 features
    df = pd.DataFrame([{
        'ph': ph,
        'Hardness': Hardness,
        'Solids': Solids,
        'Chloramines': Chloramines,
        'Sulfate': Sulfate,
        'Conductivity': Conductivity,
        'Organic_carbon': Organic_carbon,
        'Trihalomethanes': Trihalomethanes,
        'Turbidity': Turbidity,
    }])


    df['ph_x_hardness']       = df['ph'] * df['Hardness']
    df['organic_x_turbidity'] = df['Organic_carbon'] * df['Turbidity']
    df['solids_ratio']        = df['Solids'] / (df['Hardness'] + 1)

    df['log_solids']         = np.log1p(df['Solids'])
    df['log_conductivity']   = np.log1p(df['Conductivity'])

    # Buckets
    df['ph_bucket'] = pd.cut(
        df['ph'],
        bins=[0, 6, 7, 8.5, 14],
        labels=[0, 1, 2, 3]
    )
    df['organic_bucket'] = pd.cut(
        df['Organic_carbon'],
        bins=4,
        labels=False
    )

    df['ph_bucket'] = pd.to_numeric(df['ph_bucket'], errors='coerce').fillna(0).astype(int)
    df['organic_bucket'] = pd.to_numeric(df['organic_bucket'], errors='coerce').fillna(0).astype(int)

    # Extra interaction features
    df['hardness_x_conductivity'] = df['Hardness'] * df['Conductivity']
    df['chloramine_x_trihalo']    = df['Chloramines'] * df['Trihalomethanes']
    df['organic_x_sulfate']       = df['Organic_carbon'] * df['Sulfate']
    df['solid_density']           = df['Solids'] / (df['Conductivity'] + 1)

    # Indexes
    df['danger_index'] = (
        df['Trihalomethanes'] * 0.25 +
        df['Chloramines']     * 0.20 +
        df['Organic_carbon']  * 0.15 +
        df['Turbidity']       * 0.10
    )

    df['quality_index'] = (
        df['ph_bucket'] +
        df['organic_bucket'] +
        df['solids_ratio']
    )

    return df[FEATURES]


In [ ]:
sample_df = build_feature_row(
    ph=7.0,
    Hardness=150.0,
    Solids=500.0,
    Chloramines=6.0,
    Sulfate=300.0,
    Conductivity=450.0,
    Organic_carbon=10.5,
    Trihalomethanes=50.0,
    Turbidity=3.5
)

X = preprocessor.transform(sample_df)

score = regressor.predict(X)[0]

potable = classifier.predict(X)[0]

probability = classifier.predict_proba(X)[0][1]  # class 1

print("Score:", score)
print("Potable:", potable)
print("Probability:", probability)


Score: 5.305031411792825
Potable: 0
Probability: 0.1938426500900052


In [ ]:
import shutil
shutil.copy("/kaggle/working/model_artifacts/classifier.joblib", ".")
shutil.copy("/kaggle/working/model_artifacts/regressor.joblib", ".")
shutil.copy("/kaggle/working/model_artifacts/preprocessor.joblib", ".")
shutil.copy("/kaggle/working/model_artifacts/data_with_scores.csv", ".")


'./data_with_scores.csv'